# モデルが自信満々に間違えた事例を調べる (エラー分析)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_classification/error_analysis.ipynb)

NLP クックブック 09-16 ／ テキスト分類

精度の数字だけでなく、どんな事例を間違えているのかを見たい。アノテーション誤りの発見にもつながる。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline

cats = ["rec.sport.baseball", "rec.sport.hockey", "sci.space", "sci.med", "comp.graphics"]
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))
pipe = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True), LogisticRegression(C=10, max_iter=1000))

# 交差検証で、すべての事例について「その事例を学習に使っていないモデル」の予測確率を得る
proba = cross_val_predict(pipe, data.data, data.target, cv=5, method="predict_proba", n_jobs=-1)
df = pd.DataFrame({"text": [" ".join(t.split())[:110] for t in data.data], "gold": data.target, "pred": proba.argmax(axis=1),
                   "p_gold": proba[np.arange(len(proba)), data.target], "p_pred": proba.max(axis=1)})
df["gold"], df["pred"] = df["gold"].map(lambda i: cats[i]), df["pred"].map(lambda i: cats[i])
print("正解率:", (df.gold == df.pred).mean().round(3))

In [ ]:
pd.set_option("display.max_colwidth", 120)
errors = df[df.gold != df.pred].sort_values("p_gold")   # 正解ラベルの確率が低い順 = 自信満々に間違えた順
errors.head(8)

In [ ]:
print("空に近い文書の割合 (誤り / 全体):", (errors.text.str.len() < 20).mean().round(2), "/", (df.text.str.len() < 20).mean().round(2))
pd.crosstab(errors.gold, errors.pred)

- 上位には「本文が空」「引用だけ」「ラベルのほうが怪しい」事例が並ぶ。モデルの改善より先にデータの掃除が必要だと分かることが多い。
- この「交差検証の予測確率でラベル誤りを探す」考え方を体系化したのが Confident Learning (`cleanlab`)。